# Sıfırdan Dil Modeli · Bölüm 4
## Gömme: sayılardan vektörlere

Geçen bölümde öyküyü pencerelere böldük; elimizde token numaralarından oluşan tablolar var. Ama bir numara yalnızca bir etikettir: 1012 ile 1013 komşu numaralar, anlamları ise alakasız olabilir. Bu defterde her numarayı bir sayı listesine, bir **vektöre** çeviren katmanı kuruyoruz: **gömme katmanı** (*embedding*). Önce altı satırlık oyuncak bir tabloyla, sonra gerçek boyda. Sonra konum gömmesini ekliyoruz ve en sonda eğitilmiş bir tabloya, serinin modeli Çırak'ınkine bakıyoruz.

Adımlar Sebastian Raschka'nın *Build a Large Language Model (From Scratch)* kitabının 2.7 ve 2.8. bölümlerini izler. Kod, kitabın açık kaynak deposundaki kodun ([rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch), Apache-2.0) Türkçe veriyle uyarlamasıdır.

Hücreleri sırayla çalıştırın: hücreye tıklayın, **Shift + Enter**'a basın.

## 0. Önceki bölümler
Önceki bölümlerde yazdığımız kod (BPE tokenizer'ı, kayan pencere) `onceki_bolumler.py` dosyasında. Onu, öyküyü ve Çırak'ın tokenizer'ını indiriyoruz:

In [ ]:
import os
import urllib.request

REPO = ("https://raw.githubusercontent.com/"
        "kocahmet1/sifirdan-dil-modeli/main/")
for path in ["onceki_bolumler.py", "veri/kasagi.txt",
             "veri/cirak_tokenizer.json"]:
    file_path = os.path.basename(path)
    if not os.path.exists(file_path):
        urllib.request.urlretrieve(REPO + path, file_path)

In [ ]:
import torch
from onceki_bolumler import load_tokenizer, create_dataloader_v1

## 1. Tablo
Altı parçalık oyuncak bir sözlükle başlayalım. Kelimemiz "evlerimizden", dört parça:

In [ ]:
toy_vocab = {0: "ki", 1: "den", 2: "ev",
             3: "ler", 4: "de", 5: "imiz"}
input_ids = torch.tensor([2, 3, 5, 1])
print([toy_vocab[i] for i in input_ids.tolist()])

Gömme katmanı, her parça için bir satır tutan bir tablodur: 6 parça × 3 sayı. Sayılar rastgele başlar; `manual_seed` her seferinde aynı rastgele sayıları üretir, böylece çıktılarınız bizimkiyle aynı olur. `requires_grad=True`: bu sayılar eğitimde değişecek.

In [ ]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

## 2. Satır seçmek
3 numaralı parçanın ("ler") vektörü, tablonun 3 numaralı satırı (sayım sıfırdan başlar):

In [ ]:
print(embedding_layer(torch.tensor([3])))

Dört parçanın hepsi birden: her numara için bir satır.

In [ ]:
print(embedding_layer(input_ids))

## 3. One-hot: aynı işin uzun yolu
Bir numarayı, yalnızca kendi yerinde 1 olan bir vektörle de yazabiliriz (*one-hot*). Bu vektörleri tabloyla çarpınca ne olur?

In [ ]:
onehot = torch.nn.functional.one_hot(input_ids, num_classes=vocab_size)
print(onehot)

In [ ]:
result = onehot.float() @ embedding_layer.weight
print(result)
print(torch.allclose(result, embedding_layer(input_ids)))

Aynı sonuç: gömme katmanı, one-hot vektörle matris çarpımının kısa yoludur. Sıfırlarla çarpmaz, doğrudan satırı alır.

## 4. Gerçek boy
Şimdi gerçek sözlük: Çırak'ın tokenizer'ı, 4.096 parça. Her parçaya 256 sayılık bir vektör veriyoruz (kitaptaki boy).

In [ ]:
tokenizer = load_tokenizer()
vocab_size = len(tokenizer.vocab)
output_dim = 256

torch.manual_seed(123)
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(token_embedding_layer.weight.shape)
print(token_embedding_layer.weight.numel())

Geçen bölümdeki `DataLoader` ile öyküden 8 pencere alıyoruz, her biri 4 token:

In [ ]:
with open("kasagi.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token ID'leri:\n", inputs)
print("\nGirdinin şekli:\n", inputs.shape)

İlk üç pencere hangi parçalar?

In [ ]:
for row in inputs[:3]:
    print([tokenizer.decode([i]) for i in row.tolist()])

Her numarayı tablodaki satırına çeviriyoruz:

In [ ]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

## 5. Konum
İlk iki pencereye bakın: birincinin son parçası ile ikincinin ilk parçası aynı, 10 (satır sonu). Tablo ikisine de aynı vektörü veriyor; parçanın nerede durduğunu bilmiyor:

In [ ]:
print(inputs[0, 3], inputs[1, 0])
print(torch.equal(token_embeddings[0, 3], token_embeddings[1, 0]))

Çözüm: her konum için de öğrenilen bir vektör, ikinci bir tablo. Konumlar 0, 1, 2, 3; `torch.arange` bu numaraları üretir.

In [ ]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(torch.arange(max_length))
print(pos_embeddings.shape)

## 6. Toplam
Her token vektörüne kendi konumunun vektörünü ekliyoruz. (8, 4, 256) ile (4, 256) toplanınca PyTorch küçük olanı 8 pencerenin her birine ayrı ayrı ekler.

In [ ]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

In [ ]:
print(torch.equal(input_embeddings[0, 3], input_embeddings[1, 0]))

## 7. Eğitilmiş bir tablo: Çırak
Bizim tablomuz rastgele. Eğitim bu sayıları nasıl değiştirir? Serinin modeli Çırak Türkçe Vikipedi'de eğitildi; gömme tablosu bizimkiyle aynı boyda: 4.096 × 256. İndirip bir `Embedding`'e yüklüyoruz:

In [ ]:
file_path = "cirak_gomme.pt"
if not os.path.exists(file_path):
    urllib.request.urlretrieve(REPO + "veri/cirak_gomme.pt", file_path)

cirak_weights = torch.load(file_path)
cirak_embedding = torch.nn.Embedding.from_pretrained(cirak_weights)
print(cirak_embedding.weight.shape)

İki vektör aynı yöne ne kadar bakıyor? **Kosinüs benzerliği**: 1 aynı yön, 0 ilgisiz. Bir kelimenin vektörüne en çok benzeyen 4 parçayı bulalım. Kelimenin başına bir boşluk ekliyoruz: cümle içindeki hâli.

In [ ]:
def neighbors(word, embedding, k=4):
    token_id = tokenizer.encode(" " + word)[0]
    vectors = embedding.weight.detach()
    sims = torch.cosine_similarity(vectors[token_id], vectors, dim=1)
    sims[token_id] = -1   # kendisini sayma
    best = torch.topk(sims, k)
    return [(tokenizer.decode([i]), round(s, 2))
            for s, i in zip(best.values.tolist(), best.indices.tolist())]

In [ ]:
print("Rastgele:", neighbors("Ocak", token_embedding_layer))
print("Çırak:   ", neighbors("Ocak", cirak_embedding))

In [ ]:
for word in ["Ankara", "iki", "Almanya"]:
    print(word, "→", neighbors(word, cirak_embedding))

## Özet
Gömme katmanı bir tablodur: her token numarası bir satır seçer, satır o parçanın vektörüdür. Aynı işi one-hot vektörle matris çarpımı da yapar; tablo onun kısa yolu. Konum gömmesi ikinci bir tablo: her konuma bir vektör. İkisinin toplamı (8, 4, 256), modelin asıl girdisi. Tablolar rastgele başlar; eğitim onları düzenler: Çırak'ın tablosunda Ocak'ın komşuları öteki aylar.

**Kendin dene**
- `neighbors`'ı kendi kelimelerinizle deneyin (tek parça olan kelimeler: Mart, beş, film, futbol …). En şaşırtıcı komşuyu yorumlara yazın.
- `max_length`'i 8, `batch_size`'ı 2 yapın: `inputs` ve `input_embeddings`'in şekli ne olur?

**Sıradaki bölüm:** Dikkat. Kelimeler birbirine bakıyor: basit öz-dikkat.

---
Kaynak: Sebastian Raschka, *Build a Large Language Model (From Scratch)*, Manning 2024, bölüm 2.7–2.8 ve depodaki "embeddings and linear layers" eki. Kod: [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (Apache-2.0), Türkçe uyarlama: Sıfırdan Dil Modeli. Metin: Ömer Seyfettin, "Kaşağı" (kamu malı). Çırak'ın tokenizer'ı ve gömme tablosu Türkçe Vikipedi metniyle (CC BY-SA 4.0) eğitildi.